In [1]:
import numpy as np 
import soundfile as sf
import librosa
import pandas as pd 
from pathlib import Path
import pickle 
import IPython.display as ipd
import sys 
sys.path.append('../../datasets/spatial_audio_pipeline/spatial_audio_util/')
import util_audio 
import soxr
import h5py
from tqdm.auto import tqdm
%matplotlib inline 
import matplotlib.pyplot as plt

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


# Make stimuli for model simulation of 2023 mono word recognition experiment

### Same as human experiment, now using every possible foreground background pairing of stimuli - reflects full set presented to participants 
Using sanity-checked examples - removed stimuli with mismatched cue and target, or noisy excerpts. 
## Wanted Conditions:
### SNRs:
-9, -6, -3, 0, 3, inf 
### Distractor conditions:
- 1-talker
- 4-talker 
- Speech-shaped noise
- Modulated Masker 
- Music
- Natural scenes  
- 8-talker babble 

Will be using foregrounds and cues from manifest:
`/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_experiment_v00/cue_and_target_manifest.pdpkl`

Stim will be moved to `/om/user/imgriff/datasets/human_word_rec_SWC_2023`

In [2]:
stim_out_path = Path('/om/user/imgriff/datasets/human_word_rec_SWC_2023')
manifest = pd.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/screened_eval_trial_manifest_new_fnames_w_transcripts.pdpkl')
manifest.shape

(1414, 38)

In [3]:
manifest.columns

Index(['orig_full_index', 'distractor_client_id', 'distractor_clip_dur_in_s',
       'distractor_clip_end_in_s', 'distractor_clip_start_in_s',
       'distractor_corpus', 'distractor_gender', 'distractor_gender_int',
       'distractor_split', 'distractor_split_int', 'distractor_sr',
       'distractor_src_fn', 'distractor_total_file_duration_in_s',
       'distractor_word', 'src_ix', 'client_id', 'clip_dur_in_s',
       'clip_end_in_s', 'clip_start_in_s', 'corpus', 'gender', 'gender_int',
       'split', 'split_int', 'sr', 'src_fn', 'total_file_duration_in_s',
       'word', 'cue_word', 'cue_src_ix', 'cue_client_id', 'cue_src_fn',
       'cue_clip_start_in_s', 'cue_clip_end_in_s', 'gender_cond_td',
       'cue_clip_dur_in_s', 'target_transcripts', 'distractor_transcripts'],
      dtype='object')

In [4]:
eg = manifest.iloc[0]

In [5]:
x, _= librosa.load(eg['distractor_src_fn'], sr=None)
x.shape

(88200,)

#### Get distractor manifest for 4 distractors condition

##### Will use paired distractor for 1-distractor condition 


In [6]:
fn_pkl_dst = '/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl'
word_dict = pickle.load(open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", 'rb'))

# manually suppliment eliminated contractions to word dict 
words = list(word_dict.keys())
for word in words:
    if "'" in word:
        print(word)
        cleaned_word = word.replace("'", "")
        word_dict[cleaned_word] = word_dict[word]
words = list(word_dict.keys())
# words = [word.replace("'", "") for word in words]
manifest_all_words = pd.read_pickle(fn_pkl_dst)

# manually 
# filter out words not in 'words' list
manifest_all_words = manifest_all_words[manifest_all_words['word'].isin(words)]
word_counts = manifest_all_words['word'].value_counts()

# distractor manifest - talkers & words not in test foreground set but still in vocab
target_vocab = manifest['word'].unique()
distractor_manifest = manifest_all_words[~manifest_all_words['word'].isin(target_vocab)]
# filter out target talkers
distractor_manifest = distractor_manifest[~distractor_manifest.client_id.isin(manifest.client_id.unique())]

can't
couldn't
didn't
doesn't
don't
isn't
let's
that's
there's
they're
wasn't
we're
what's
won't
you'll
you're


In [7]:
manifest['word_int'] = manifest['word'].map(word_dict)
manifest['dist_word_int'] = manifest['distractor_word'].map(word_dict)

## We will store target and distractor excerpts separately in the same h5 file

Store cues, foregrounds, and distractors as separate datagroups in the same file - will select the distractor key based on the test. 
Allows for SNR combination on the fly 

In [8]:
sys.path.append('/om2/user/imgriff/datasets/spatial_audio_pipeline/spatial_audio_util/')
import util_audio 

In [9]:
util_audio.get_excerpt

<function util_audio.get_excerpt(dfi, dur=3.0, sr=50000, pad_with_context=True, jitter_fraction=0)>

In [10]:
def get_avg_f0(sig, SR, fmin=80, fmax=300):
    # get talker f0 - bound to range of human voice 
    target_f0, voice_flag, voice_probs = librosa.pyin(sig, fmin=80, fmax=300, sr=SR, fill_na=np.nan)
    target_f0 = target_f0[voice_flag]        
    avg_target_f0 = np.nanmean(target_f0)
    return avg_target_f0

In [11]:
## Set up parameters of dataset 

SR = 44100
dBSPL = 60 
# timing in seconds 
signal_dur = 2.5 # add context to crop to 2s post cochleagram 
win_dur = 10 # ms 

### init output h5 file 
stim_out_path = Path('/om/user/imgriff/datasets/human_word_rec_SWC_2023/')

output_file_name = stim_out_path / 'model_eval_stim.h5'

# init output key list
signal_keys  = ['target_signal', 'cue_signal', 'one_dist_signal',
         'four_dist_signal', 'babble_dist_signal', 'music_dist_signal',
         'nat_dist_signal', 'ssn_dist_signal', 'modulated_dist_signal']

label_keys = ['word_int_label', 'confusion_int_label', 'target_f0', 'cue_f0', 'one_dist_f0']

signal_len = int(SR * signal_dur)
n_examples = len(manifest)

In [12]:
output_file_name

PosixPath('/om/user/imgriff/datasets/human_word_rec_SWC_2023/model_eval_stim.h5')

In [13]:
np.random.seed(0)

# init background stim selections 
path_to_bg_stim = Path("/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/")

conditions = ['background_musdb18hq',
              'background_cv08talkerbabble',
              # 'background_issnstationary',
              # 'background_issnfestenplomp',
            #   'background_audioset',
              'background_ieeeaaspcasa']

bg_stim = {}
for condition in conditions: 
    bg_stim[condition] = np.random.choice(list((path_to_bg_stim / condition).glob('*.wav')), n_examples, replace=True)

with h5py.File(output_file_name, 'w') as f:
    for key in signal_keys:
        f.create_dataset(key, shape=[n_examples, signal_len], dtype=np.float32)
    for key in label_keys:
        if 'f0' in key:
            f.create_dataset(key, shape=[n_examples], dtype=np.float32)
        else:
            f.create_dataset(key, shape=[n_examples], dtype=np.int32)
    # Read, cut and save the cue, target and distractor signals
    for ix, row in enumerate(tqdm(manifest.itertuples(), total=n_examples)):
        # save labels 
        f['word_int_label'][ix] = row.word_int
        f['confusion_int_label'][ix] = row.dist_word_int
        # read in the audio files and store - these are pre cut 
        target_signal, _ = librosa.load(row.src_fn, sr=SR)
        cue_signal, _ = librosa.load(row.cue_src_fn, sr=SR)
        one_dist_signal, _ = librosa.load(row.distractor_src_fn, sr=SR)        
        # normalize the signals
        target_signal = util_audio.set_dBSPL(target_signal, dBSPL) 
        cue_signal = util_audio.set_dBSPL(cue_signal, dBSPL)
        one_dist_signal = util_audio.set_dBSPL(one_dist_signal, dBSPL)
        # signals read in at 2 seconds - padd up to 2.5 seconds - will be cropped later
        f['target_signal'][ix] = util_audio.pad_or_trim_to_len(target_signal, signal_len, mode='both')
        f['cue_signal'][ix] = util_audio.pad_or_trim_to_len(cue_signal, signal_len, mode='both')
        f['one_dist_signal'][ix] = util_audio.pad_or_trim_to_len(one_dist_signal, signal_len, mode='both')
        # get f0 traces for each
        target_f0 = get_avg_f0(target_signal, SR)
        cue_f0 = get_avg_f0(cue_signal, SR)
        one_dist_f0 = get_avg_f0(one_dist_signal, SR)
        # store
        f['target_f0'][ix] = target_f0
        f['cue_f0'][ix] = cue_f0
        f['one_dist_f0'][ix] = one_dist_f0

        # gen 4-distractor samples
        bg_examples = distractor_manifest.sample(4)
        bg_audio = bg_examples.apply(lambda x: util_audio.get_excerpt(x, dur=signal_dur, sr=SR, pad_with_context=True, jitter_fraction=0), axis=1)
        four_dist_signal = np.sum(bg_audio, axis=0)
        f['four_dist_signal'][ix] = util_audio.pad_or_trim_to_len(
                                                util_audio.set_dBSPL(four_dist_signal, dBSPL),
                                                signal_len, mode='both')

        # gen ssn distractor
        ssn_signal = util_audio.spectrally_matched_noise(target_signal, SR)
        f['ssn_dist_signal'][ix] = util_audio.pad_or_trim_to_len(
                                        util_audio.set_dBSPL(ssn_signal, dBSPL),
                                        signal_len, mode='both')

        # gen modulated distractor 
        modulated_signal = util_audio.festen_plomp_fluctuating_noise(one_dist_signal, ssn_signal, sr=SR, dbspl=dBSPL)
        f['modulated_dist_signal'][ix] = util_audio.pad_or_trim_to_len(
                                                util_audio.set_dBSPL(modulated_signal, dBSPL), 
                                                signal_len, mode='both')
    
        # get music distractor
        music_signal, _ = librosa.load(bg_stim['background_musdb18hq'][ix], sr=SR)
        music_signal = util_audio.set_dBSPL(music_signal, dBSPL)
        f['music_dist_signal'][ix] = util_audio.pad_or_trim_to_len(music_signal, signal_len, mode='both')

        # get babble distractor
        babble_signal, _ = librosa.load(bg_stim['background_cv08talkerbabble'][ix], sr=SR)
        babble_signal = util_audio.set_dBSPL(babble_signal, dBSPL)
        f['babble_dist_signal'][ix] = util_audio.pad_or_trim_to_len(babble_signal, signal_len, mode='both')

        # get natural scene distractor
        nat_signal, _ = librosa.load(bg_stim['background_ieeeaaspcasa'][ix], sr=SR)
        nat_signal = util_audio.set_dBSPL(nat_signal, dBSPL)
        f['nat_dist_signal'][ix] = util_audio.pad_or_trim_to_len(nat_signal, signal_len, mode='both')
        

        





  0%|          | 0/1414 [00:00<?, ?it/s]

/tmp/ipykernel_2395453/2164091652.py:5: RuntimeWarning: Mean of empty slice
  avg_target_f0 = np.nanmean(target_f0)
/tmp/ipykernel_2395453/2164091652.py:5: RuntimeWarning: Mean of empty slice
  avg_target_f0 = np.nanmean(target_f0)
/tmp/ipykernel_2395453/2164091652.py:5: RuntimeWarning: Mean of empty slice
  avg_target_f0 = np.nanmean(target_f0)
/tmp/ipykernel_2395453/2164091652.py:5: RuntimeWarning: Mean of empty slice
  avg_target_f0 = np.nanmean(target_f0)
/tmp/ipykernel_2395453/2164091652.py:5: RuntimeWarning: Mean of empty slice
  avg_target_f0 = np.nanmean(target_f0)
/tmp/ipykernel_2395453/2164091652.py:5: RuntimeWarning: Mean of empty slice
  avg_target_f0 = np.nanmean(target_f0)
/tmp/ipykernel_2395453/2164091652.py:5: RuntimeWarning: Mean of empty slice
  avg_target_f0 = np.nanmean(target_f0)
/tmp/ipykernel_2395453/2164091652.py:5: RuntimeWarning: Mean of empty slice
  avg_target_f0 = np.nanmean(target_f0)
/tmp/ipykernel_2395453/2164091652.py:5: RuntimeWarning: Mean of empty sl

In [ ]:
distractor_eg.distractor_word

'studio'

In [19]:
## Check file 
### init output h5 file 
stim_out_path = Path('/om/user/imgriff/datasets/human_word_rec_SWC_2023/')

manifest_out_name = "/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/screened_eval_trial_manifest_new_fnames_w_transcripts_and_f0.pdpkl"
manifest = pd.read_pickle(manifest_out_name)

output_file_name = stim_out_path / 'model_eval_stim.h5'
h5_file = h5py.File(output_file_name, 'r')

In [3]:
h5_file.keys()

<KeysViewHDF5 ['babble_dist_signal', 'confusion_int_label', 'cue_f0', 'cue_signal', 'four_dist_signal', 'modulated_dist_signal', 'music_dist_signal', 'nat_dist_signal', 'one_dist_f0', 'one_dist_signal', 'ssn_dist_signal', 'target_f0', 'target_signal', 'word_int_label']>

In [14]:
for key in h5_file.keys():
    if 'f0' in key:
        print(key, h5_file[key].shape)
        data = h5_file[key][:]
        ixs = np.where(np.logical_or(data > 300 , data < 80))[0]
        if len(ixs) > 0:
            print(key)


cue_f0 (1414,)
one_dist_f0 (1414,)
target_f0 (1414,)


In [20]:
# Add f0s to manifest and save 
manifest['target_f0'] = h5_file['target_f0'][:]
manifest['cue_f0'] = h5_file['cue_f0'][:]
manifest['distractor_f0'] = h5_file['one_dist_f0'][:]

In [22]:
h5_file.close()

In [23]:
manifest_out_name = "/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/human_attn_experiment_v00/screened_eval_trial_manifest_new_fnames_w_transcripts_and_f0.pdpkl"
manifest.to_pickle(manifest_out_name)

### Dev dataset class

In [61]:
import importlib 
import corpus.swc_mono_test as swc_corpus

In [72]:
importlib.reload(swc_corpus)

keys = ['one_distractor', 'four_distractor', 'stationary', 'modulated_distractor',
        'music', 'babble', 'natural_scene']
for key in keys:
    dataset = swc_corpus.SWCMonoTestSetH5Dataset(output_file_name, key, 44_100)
    for _ in tqdm(dataset):
        continue 

  0%|          | 0/1414 [00:00<?, ?it/s]

  0%|          | 0/1414 [00:00<?, ?it/s]

  0%|          | 0/1414 [00:00<?, ?it/s]

  0%|          | 0/1414 [00:00<?, ?it/s]

  0%|          | 0/1414 [00:00<?, ?it/s]

  0%|          | 0/1414 [00:00<?, ?it/s]

  0%|          | 0/1414 [00:00<?, ?it/s]